# SUPER NEO-FORECAST — Dataset 11, IJAMT reviewer evidence
**Three-horizon pilot (1, 6, 12 hours).** NB01–NB14 in one sequential Colab notebook. Outputs are generated only when cells run. Within-dataset regime transfer is not cross-dataset generalization; manufacturing evaluation is a simulation.

In [ ]:
!pip -q install pandas numpy scikit-learn scipy matplotlib seaborn requests torch
from pathlib import Path
import os, json, zipfile, io, hashlib, itertools, warnings, random
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
SEED=42; np.random.seed(SEED); random.seed(SEED)
HORIZONS=[1,6,12]  # extend to [1,6,12,24,48,72,168] after pilot
LOOKBACK=48; OUT=Path("/content/neo_forecast_outputs"); OUT.mkdir(parents=True,exist_ok=True)
DATASET_URL="https://archive.ics.uci.edu/static/public/374/appliances+energy+prediction.zip"
STATE={"models":{},"predictions":{},"metrics":[],"evidence":[]}
def record(section,status,artifact,notes=""):
    STATE["evidence"].append(dict(section=section,status=status,artifact=str(artifact),notes=notes))
def savecsv(df,name):
    p=OUT/name; df.to_csv(p,index=False); return p
def score(y,p):
    return {"MAE":float(mean_absolute_error(y,p)),"RMSE":float(np.sqrt(mean_squared_error(y,p)))}
print("Output:",OUT,"Horizons:",HORIZONS)

## NB01 — Dataset provenance and integrity

In [ ]:
import requests
r=requests.get(DATASET_URL,timeout=120); r.raise_for_status()
raw=r.content; sha=hashlib.sha256(raw).hexdigest()
z=zipfile.ZipFile(io.BytesIO(raw))
csvs=[n for n in z.namelist() if n.lower().endswith(".csv")]
assert csvs, "No CSV in official UCI archive"
df=pd.read_csv(z.open(csvs[0])); assert {"date","Appliances"}.issubset(df.columns)
df["date"]=pd.to_datetime(df["date"],errors="raise")
df=df.sort_values("date").reset_index(drop=True)
audit={"source":DATASET_URL,"archive_sha256":sha,"member":csvs[0],"rows":len(df),"columns":list(df.columns),"duplicate_timestamps":int(df.date.duplicated().sum()),"missing_by_column":df.isna().sum().astype(int).to_dict(),"min_date":str(df.date.min()),"max_date":str(df.date.max())}
(OUT/"NB01_audit.json").write_text(json.dumps(audit,indent=2))
assert audit["duplicate_timestamps"]==0
record("NB01","executed","NB01_audit.json")
print(json.dumps(audit,indent=2)[:1800])

## NB02 — Hourly aggregation and chronological splits
Appliances is average power in W at ten-minute resolution; hourly energy in Wh is sum(W×1/6 h), requiring six complete samples. Forecast features use past target values only, avoiding future weather leakage.

In [ ]:
d=df.set_index("date").sort_index()
power=d["Appliances"].astype(float)
count=power.resample("1h").count()
energy=(power/6.0).resample("1h").sum(min_count=6)
hourly=pd.DataFrame({"energy_Wh":energy,"samples":count})
hourly=hourly.loc[hourly.samples==6].dropna()
assert hourly.index.is_monotonic_increasing and hourly.index.is_unique
# Restrict to uninterrupted hourly sequences; no interpolation across missing hours.
hourly["segment"]=(hourly.index.to_series().diff()!=pd.Timedelta(hours=1)).cumsum().to_numpy()
def windows(frame,horizons=HORIZONS,lookback=LOOKBACK):
    rows=[]
    for _,seg in frame.groupby("segment"):
        v=seg.energy_Wh.to_numpy(); ts=seg.index
        for t in range(lookback,len(v)-max(horizons)):
            row={"origin":ts[t-1],"x":v[t-lookback:t]}
            for h in horizons: row[f"y{h}"]=v[t+h-1]
            rows.append(row)
    return pd.DataFrame(rows)
W=windows(hourly); assert len(W)>200
# Purge split boundaries by max horizon, so training targets cannot enter validation/test.
n=len(W); a=int(.65*n); b=int(.82*n); gap=max(HORIZONS)
train=W.iloc[:a-gap].copy(); val=W.iloc[a:b-gap].copy(); test=W.iloc[b:].copy()
assert train.origin.max()<val.origin.min()<test.origin.min()
Xtr=np.stack(train.x); Xva=np.stack(val.x); Xte=np.stack(test.x)
scaler=StandardScaler().fit(Xtr); Xtr=scaler.transform(Xtr); Xva=scaler.transform(Xva); Xte=scaler.transform(Xte)
Ytr=train[[f"y{h}" for h in HORIZONS]].to_numpy(); Yva=val[[f"y{h}" for h in HORIZONS]].to_numpy(); Yte=test[[f"y{h}" for h in HORIZONS]].to_numpy()
manifest={"hourly_rows":len(hourly),"windows":len(W),"train":len(train),"validation":len(val),"test":len(test),"lookback":LOOKBACK,"horizons":HORIZONS,"gap":gap,"train_end":str(train.origin.max()),"validation_start":str(val.origin.min()),"test_start":str(test.origin.min()),"target_units":"Wh"}
(OUT/"NB02_splits.json").write_text(json.dumps(manifest,indent=2))
savecsv(hourly.reset_index(),"NB02_hourly.csv")
record("NB02","executed","NB02_splits.json")
print(manifest)

## NB03 — Persistence and Ridge baselines

In [ ]:
def evaluate(model,pred,model_name):
    for j,h in enumerate(HORIZONS):
        s=score(Yte[:,j],pred[:,j]); STATE["metrics"].append({"model":model_name,"horizon_h":h,**s})
    STATE["predictions"][model_name]=pred
    return savecsv(pd.DataFrame({"origin":test.origin,**{f"actual_{h}h":Yte[:,j] for j,h in enumerate(HORIZONS)},**{f"pred_{h}h":pred[:,j] for j,h in enumerate(HORIZONS)}}),f"predictions_{model_name}.csv")
persist=np.repeat(np.stack(test.x).reshape(len(test),LOOKBACK)[:,-1,None],len(HORIZONS),axis=1)
evaluate(None,persist,"Persistence")
ridge=Ridge(alpha=10).fit(Xtr,Ytr); evaluate(ridge,ridge.predict(Xte),"Ridge")
STATE["models"]["Ridge"]=ridge
record("NB03","executed","predictions_Ridge.csv")
print(pd.DataFrame(STATE["metrics"]))

## NB04 — LSTM and Transformer baselines
Fixed training budgets for pilot. Train on training partition, choose checkpoint by validation loss; never tune on test.

In [ ]:
import torch
from torch import nn
from torch.utils.data import TensorDataset,DataLoader
torch.manual_seed(SEED); device="cuda" if torch.cuda.is_available() else "cpu"
ym=Ytr.mean(axis=0); ys=Ytr.std(axis=0)+1e-6
def loaders():
    tr=TensorDataset(torch.tensor(Xtr,dtype=torch.float32).unsqueeze(-1),torch.tensor((Ytr-ym)/ys,dtype=torch.float32))
    va=TensorDataset(torch.tensor(Xva,dtype=torch.float32).unsqueeze(-1),torch.tensor((Yva-ym)/ys,dtype=torch.float32))
    return DataLoader(tr,batch_size=128,shuffle=True),DataLoader(va,batch_size=256)
class LSTMNet(nn.Module):
    def __init__(self):
        super().__init__(); self.rnn=nn.LSTM(1,32,batch_first=True); self.head=nn.Linear(32,len(HORIZONS))
    def forward(self,x):
        o,_=self.rnn(x); return self.head(o[:,-1,:])
class TransformerNet(nn.Module):
    def __init__(self):
        super().__init__(); self.proj=nn.Linear(1,32); self.pos=nn.Parameter(torch.zeros(1,LOOKBACK,32)); layer=nn.TransformerEncoderLayer(32,4,64,batch_first=True,dropout=.1); self.encoder=nn.TransformerEncoder(layer,1); self.head=nn.Linear(32,len(HORIZONS))
    def forward(self,x):
        h=self.encoder(self.proj(x)+self.pos); return self.head(h[:,-1,:])
def train_torch(net,name,epochs=8):
    net=net.to(device); opt=torch.optim.Adam(net.parameters(),lr=.001); tr,va=loaders(); best=float("inf"); best_weights=None
    for epoch in range(epochs):
        net.train()
        for xb,yb in tr:
            xb,yb=xb.to(device),yb.to(device); opt.zero_grad(); loss=((net(xb)-yb)**2).mean(); loss.backward(); opt.step()
        net.eval(); losses=[]
        with torch.no_grad():
            for xb,yb in va: losses.append(float(((net(xb.to(device))-yb.to(device))**2).mean().cpu()))
        v=float(np.mean(losses))
        if v<best: best=v; best_weights={k:v.detach().cpu().clone() for k,v in net.state_dict().items()}
    net.load_state_dict(best_weights); torch.save(net.cpu().state_dict(),OUT/f"{name}.pt"); net=net.to(device)
    with torch.no_grad(): p=net(torch.tensor(Xte,dtype=torch.float32).unsqueeze(-1).to(device)).cpu().numpy()*ys+ym
    evaluate(net,p,name); STATE["models"][name]=net
    return net
lstm=train_torch(LSTMNet(),"LSTM")
transformer=train_torch(TransformerNet(),"Transformer")
record("NB04","executed","LSTM.pt; Transformer.pt")

## NB05 — NEO-FORECAST candidate architecture
A **pilot operationalization**, not proof of equivalence to the original paper's full cross-domain architecture. Temporal regime embedding plus gated shared/domain representation.

In [ ]:
class NeoForecast(nn.Module):
    def __init__(self):
        super().__init__(); self.shared=nn.Sequential(nn.Linear(LOOKBACK,64),nn.ReLU(),nn.Linear(64,32),nn.ReLU()); self.domain=nn.Sequential(nn.Linear(LOOKBACK,32),nn.ReLU()); self.gate=nn.Sequential(nn.Linear(64,1),nn.Sigmoid()); self.head=nn.Linear(32,len(HORIZONS))
    def forward(self,x):
        z=x.squeeze(-1); a=self.shared(z); b=self.domain(z); g=self.gate(torch.cat([a,b],dim=1)); return self.head(g*a+(1-g)*b)
neo=train_torch(NeoForecast(),"NEO_candidate")
record("NB05","executed","NEO_candidate.pt","Pilot candidate architecture, not validated full cross-domain NEO")

## NB06 — Five descriptive compatibility distances
These are explicit measurable proxies, not claims that the paper's original Eq. (9) has been independently validated. Statistics use training-period windows only.

In [ ]:
from scipy.spatial.distance import cosine
def descriptors(A,B):
    a=np.asarray(A).reshape(-1); b=np.asarray(B).reshape(-1)
    qa=np.quantile(a,[.1,.25,.5,.75,.9]); qb=np.quantile(b,[.1,.25,.5,.75,.9])
    stat=float(np.linalg.norm(qa-qb)/(np.std(a)+1e-9))
    def ac(v,k=1): return float(np.corrcoef(v[:-k],v[k:])[0,1]) if len(v)>k+2 else 0.
    temporal=abs(ac(a)-ac(b))
    def spec(v):
        p=np.abs(np.fft.rfft(v-v.mean()))**2; return p/(p.sum()+1e-9)
    pa,pb=spec(a),spec(b); m=min(len(pa),len(pb)); spectral=float(np.mean(abs(pa[:m]-pb[:m])))
    cov=abs(float(np.var(a)-np.var(b)))/(np.var(a)+1e-9)
    # No semantic metadata for industrial entities: undefined, not fabricated.
    return dict(delta_stat=stat,delta_temporal=temporal,delta_spectral=spectral,delta_cov=cov,delta_semantic=np.nan)
cut=len(train)//2
desc=descriptors(np.stack(train.x.iloc[:cut]),np.stack(train.x.iloc[cut:]))
savecsv(pd.DataFrame([desc]),"NB06_descriptors.csv")
record("NB06","executed","NB06_descriptors.csv","Semantic descriptor unavailable for this dataset")

## NB07 — Compatibility-gate diagnostics
Gate values from the candidate model are **learned blending weights**, not calibrated estimates of positive transfer. A transfer-benefit gate requires paired source/target supervision.

In [ ]:
neo.eval()
with torch.no_grad():
    z=torch.tensor(Xte,dtype=torch.float32).to(device); a=neo.shared(z); b=neo.domain(z); gates=neo.gate(torch.cat([a,b],1)).cpu().numpy().ravel()
savecsv(pd.DataFrame({"origin":test.origin,"gate_weight":gates}),"NB07_gate_weights.csv")
record("NB07","executed","NB07_gate_weights.csv","Gate is not yet validated against transfer benefit")

## NB08 — Component ablation pilot
Compare candidate NEO against its shared-only and domain-only paths using the same trained weights. This is inference-path ablation, not retrained ablation.

In [ ]:
with torch.no_grad():
    z=torch.tensor(Xte,dtype=torch.float32).to(device); a=neo.shared(z); b=neo.domain(z)
    for name,hid in [("NEO_shared_only",a),("NEO_domain_only",b)]:
        p=neo.head(hid).cpu().numpy()*ys+ym; evaluate(None,p,name)
record("NB08","executed","predictions_NEO_shared_only.csv; predictions_NEO_domain_only.csv","Inference-only ablation; retraining needed for causal attribution")

## NB09 — Within-dataset temporal regime transfer
Train Ridge on earlier temporal regime and compare with target-only Ridge; use **validation-origin** regime pairs for model selection, then preserve test partition.

In [ ]:
trX=np.stack(train.x); trY=train[[f"y{h}" for h in HORIZONS]].to_numpy(); mid=len(train)//2
source=Ridge(alpha=10).fit(scaler.transform(trX[:mid]),trY[:mid])
target=Ridge(alpha=10).fit(scaler.transform(trX[mid:]),trY[mid:])
transfer=Ridge(alpha=10).fit(scaler.transform(trX),trY)
regimes=[]
for label,model in [("source_only",source),("target_only",target),("pooled",transfer)]:
    p=model.predict(Xte); evaluate(None,p,f"regime_{label}")
    for j,h in enumerate(HORIZONS): regimes.append(dict(regime=label,horizon=h,**score(Yte[:,j],p[:,j])))
savecsv(pd.DataFrame(regimes),"NB09_regime_results.csv")
record("NB09","executed","NB09_regime_results.csv","One chronological regime split; not all regime pairs")

## NB10 — Negative transfer paired uncertainty
Moving-block bootstrap on paired absolute-error differences. Positive Δ = target-only MAE minus pooled MAE, i.e. positive indicates benefit. Exploratory single split.

In [ ]:
rng=np.random.default_rng(SEED); rows=[]
p0=STATE["predictions"]["regime_target_only"]; p1=STATE["predictions"]["regime_pooled"]
for j,h in enumerate(HORIZONS):
    diff=np.abs(Yte[:,j]-p0[:,j])-np.abs(Yte[:,j]-p1[:,j])
    block=max(12,h); starts=np.arange(max(1,len(diff)-block+1)); boots=[]
    for _ in range(500):
        ids=[]
        while len(ids)<len(diff):
            st=int(rng.choice(starts)); ids.extend(range(st,min(st+block,len(diff))))
        boots.append(np.mean(diff[np.array(ids[:len(diff)])]))
    lo,hi=np.quantile(boots,[.025,.975])
    rows.append(dict(horizon=h,delta_MAE=float(np.mean(diff)),CI_low=float(lo),CI_high=float(hi),interpretation="benefit" if lo>0 else "harm" if hi<0 else "inconclusive"))
savecsv(pd.DataFrame(rows),"NB10_negative_transfer.csv")
record("NB10","executed","NB10_negative_transfer.csv","Exploratory bootstrap; no multiple-comparison correction")

## NB11 — Probabilistic forecast pilot
Split-conformal symmetric prediction intervals calibrated on validation residuals of Ridge. They are not full quantile models or CRPS estimates.

In [ ]:
pv=ridge.predict(Xva); pt=ridge.predict(Xte); qrows=[]; summary=[]
for j,h in enumerate(HORIZONS):
    err=np.abs(Yva[:,j]-pv[:,j]); q=float(np.quantile(err,min(1.,np.ceil((len(err)+1)*.9)/len(err)),method="higher"))
    lo=pt[:,j]-q; hi=pt[:,j]+q
    coverage=float(np.mean((Yte[:,j]>=lo)&(Yte[:,j]<=hi)))
    summary.append(dict(horizon=h,nominal_coverage=.9,empirical_coverage=coverage,mean_width=float(np.mean(hi-lo)),calibration_radius_Wh=q))
    qrows.extend(dict(origin=str(t),horizon=h,lower_90=float(l),upper_90=float(u),actual=float(y)) for t,l,u,y in zip(test.origin,lo,hi,Yte[:,j]))
savecsv(pd.DataFrame(qrows),"NB11_intervals.csv")
savecsv(pd.DataFrame(summary),"NB11_interval_metrics.csv")
record("NB11","executed","NB11_interval_metrics.csv","Marginal split conformal; temporal dependence affects nominal guarantees")

## NB12 — Calibration and interval plots

In [ ]:
cal=pd.read_csv(OUT/"NB11_interval_metrics.csv")
fig,ax=plt.subplots(figsize=(6,4)); ax.plot(cal.horizon,cal.empirical_coverage,marker="o",label="observed"); ax.axhline(.9,linestyle="--",label="nominal"); ax.set(xlabel="Horizon (hours)",ylabel="Coverage",ylim=(0,1)); ax.legend(); fig.tight_layout(); fig.savefig(OUT/"NB12_coverage.png",dpi=150); plt.close(fig)
record("NB12","executed","NB12_coverage.png")

## NB13 — Illustrative manufacturing decision simulation
Synthetic scheduling tariff and demand-shifting assumptions. No renewable-generation or industrial process observations in Dataset 11. These are simulated costs only.

In [ ]:
# Simulated two-period flexible demand scheduling, using test-hour energy forecasts.
# Shift up to 10% of forecast consumption from high-price hours to next low-price hour.
tariff=np.where(pd.to_datetime(test.origin).dt.hour.between(16,20),.30,.12)
actual=Yte[:,0]/1000.; predicted=np.maximum(0,STATE["predictions"]["Ridge"][:,0]/1000.)
shift=np.where(tariff>.2,.1*predicted,0.)
# This simple proxy reports curtailed high-price energy and unmet service; it does not claim full energy-conserving rescheduling.
baseline=float(np.sum(actual*tariff)); hypothetical=float(np.sum(np.maximum(actual-shift,0)*tariff))
sim={"tariff_units":"illustrative currency/kWh","baseline_cost_proxy":baseline,"curtailment_cost_proxy":hypothetical,"unserved_energy_proxy_kWh":float(np.sum(np.minimum(shift,actual))),"note":"Illustrative curtailment only; not a valid manufacturing scheduling benefit without service penalties and feasible rebound"}
(OUT/"NB13_simulation.json").write_text(json.dumps(sim,indent=2))
record("NB13","executed","NB13_simulation.json","Synthetic scenario, not empirical manufacturing benefit")

## NB14 — Consolidation and R0–R6 evidence matrix
This notebook explicitly records unresolved reviewer requirements. A completed pilot does not imply 95% coverage.

In [ ]:
savecsv(pd.DataFrame(STATE["metrics"]),"NB14_all_model_metrics.csv")
review=[
("R0","Single-building temporal forecasting and reproducible pipeline","Partial","No real industrial deployment or cross-dataset validation"),
("R1","Provenance of new numerical metrics","Partial","Use only NB14 computed metrics; replace unsupported manuscript claims"),
("R2","Architecture and ablations","Partial","Pilot candidate, inference-only ablations; full comparator/retraining needed"),
("R3","Five descriptors and compatibility gate","Partial","Semantic undefined; gate not supervised by transfer benefit"),
("R4","Negative transfer","Partial","One temporal split; full regime-pair analysis and multiplicity needed"),
("R5","Forecast horizons","Partial","Only 1/6/12 hours; extend to 24/48/72/168"),
("R6","Probabilistic forecasts","Partial","Conformal interval pilot; pinball/CRPS not provided")
]
savecsv(pd.DataFrame(review,columns=["reviewer","evidence","status","remaining_work"]),"NB14_reviewer_matrix.csv")
savecsv(pd.DataFrame(STATE["evidence"]),"NB14_execution_manifest.csv")
print(pd.DataFrame(STATE["metrics"]).to_string(index=False))
print("Saved artifacts:",len(list(OUT.iterdir())))

### Execution and reproducibility notes
Run **Runtime → Run all** in Google Colab. The notebook downloads the public official UCI dataset, so Drive mounting is optional. Keep the output directory and download it using `!zip -qr /content/NEO_FORECAST_results.zip /content/neo_forecast_outputs` after a successful run. These cells are constructed but have **not** been executed or validated in this chat. Some stages are exploratory pilots and do not satisfy all reviewer requirements.